# EuroSAT Experiments (Terminal-First)

This notebook is command-first and does not require editing strategy variables.
Run only the cell(s) you need.

In [1]:
import sys
from pathlib import Path
import csv

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent
%cd {ROOT}

OUT_BASE = ROOT / "outputs" / "eurosat_mobilenetv2"
OUT_BASE.mkdir(parents=True, exist_ok=True)

print("python executable:", sys.executable)
print("project root:", ROOT)

E:\ANU\COMP6242-group-project
python executable: e:\ANU\COMP6242-group-project\.venv\Scripts\python.exe
project root: E:\ANU\COMP6242-group-project


In [2]:
import torch
print("torch=", torch.__version__)
print("cuda build=", torch.version.cuda)
print("cuda available=", torch.cuda.is_available())
print("gpu=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch= 2.6.0+cu124
cuda build= 12.4
cuda available= True
gpu= NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
# Data preparation with safety check
import subprocess
import os
from copy import deepcopy
dataset_dir = ROOT / "data" / "eurosat" / "2750"
metadata_csv = ROOT / "data" / "metadata.csv"
if dataset_dir.exists() and metadata_csv.exists():
    print(f"Dataset already prepared: {dataset_dir}")
    print(f"Metadata exists: {metadata_csv}")
else:
    print("Dataset/metadata missing, preparing now...")
    subprocess.run([sys.executable, "scripts/prepare_eurosat.py", "--root", "data", "--download", "--out_csv", "data/metadata.csv"], check=True)
    print("Preparation done.")

def stream_cmd(cmd: list[str]) -> None:
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed ({ret}): {' '.join(cmd)}")

os.environ["EUROSAT_FORCE_PROGRESS"] = "1"
sys.path.insert(0, str((ROOT / "src").resolve()))
from eurosat_baseline.config import Config, load_config
from eurosat_baseline.train import train_main

def run_strategy(name: str) -> dict:
    base = load_config("configs/base.yaml")
    raw = deepcopy(base.raw)
    raw["training"]["strategy"] = name
    raw["output_dir"] = f"outputs/eurosat_mobilenetv2/{name}"
    raw.setdefault("runtime", {})["force_progress"] = True
    artifacts = train_main(Config(raw=raw), dummy=False)
    print("best checkpoint:", artifacts.best_ckpt)
    print("summary:", artifacts.summary_json)
    d = json.loads(Path(artifacts.summary_json).read_text(encoding="utf-8"))
    row = {
        "strategy": name,
        "best_val_top1": d.get("best_val_top1"),
        "test_top1_acc": d.get("test_top1_acc"),
        "test_macro_f1": d.get("test_macro_f1"),
        "train_seconds": d.get("train_seconds"),
        "trainable_params": d.get("trainable_params"),
        "total_params": d.get("total_params"),
    }
    print("row:", row)
    return row

Dataset already prepared: E:\ANU\COMP6242-group-project\data\eurosat\2750
Metadata exists: E:\ANU\COMP6242-group-project\data\metadata.csv


## One-Click Strategy Cells
Each cell runs in-process with live progress bars and requires no code edits.

In [4]:
# zero_shot
run_strategy("zero_shot")

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=2.2913 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=2 train_loss=2.2915 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=3 train_loss=2.2924 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=4 train_loss=2.2916 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=5 train_loss=2.2920 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=6 train_loss=2.2920 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=7 train_loss=2.2928 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=8 train_loss=2.2922 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


strategy=zero_shot test_top1=0.1582 test_f1=0.1154 time=544.4s
best checkpoint: outputs\eurosat_mobilenetv2\zero_shot\best.pt
summary: outputs\eurosat_mobilenetv2\zero_shot\summary.json


In [5]:
# from_scratch
run_strategy("from_scratch")

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=1.0141 val_loss=0.8597 val_top1=0.7047 val_f1=0.6843


epoch=2 train_loss=0.7052 val_loss=0.5964 val_top1=0.7894 val_f1=0.7818


epoch=3 train_loss=0.5928 val_loss=0.4877 val_top1=0.8315 val_f1=0.8252


epoch=4 train_loss=0.5050 val_loss=0.3761 val_top1=0.8681 val_f1=0.8638


epoch=5 train_loss=0.4236 val_loss=0.2872 val_top1=0.9026 val_f1=0.8999


epoch=6 train_loss=0.3598 val_loss=0.2560 val_top1=0.9023 val_f1=0.8987


epoch=7 train_loss=0.3238 val_loss=0.2031 val_top1=0.9329 val_f1=0.9310


epoch=8 train_loss=0.2777 val_loss=0.2318 val_top1=0.9166 val_f1=0.9147


strategy=from_scratch test_top1=0.9216 test_f1=0.9193 time=982.8s
best checkpoint: outputs\eurosat_mobilenetv2\from_scratch\best.pt
summary: outputs\eurosat_mobilenetv2\from_scratch\summary.json


In [6]:
# linear_probe
run_strategy("linear_probe")

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=1.1781 val_loss=0.7717 val_top1=0.8252 val_f1=0.8152


epoch=2 train_loss=0.6694 val_loss=0.5597 val_top1=0.8537 val_f1=0.8466


epoch=3 train_loss=0.5595 val_loss=0.4751 val_top1=0.8704 val_f1=0.8641


epoch=4 train_loss=0.5093 val_loss=0.4333 val_top1=0.8755 val_f1=0.8695


epoch=5 train_loss=0.4760 val_loss=0.3835 val_top1=0.8832 val_f1=0.8782


epoch=6 train_loss=0.4527 val_loss=0.3773 val_top1=0.8859 val_f1=0.8816


epoch=7 train_loss=0.4380 val_loss=0.3490 val_top1=0.8904 val_f1=0.8861


epoch=8 train_loss=0.4237 val_loss=0.3485 val_top1=0.8945 val_f1=0.8896


strategy=linear_probe test_top1=0.8942 test_f1=0.8901 time=557.5s
best checkpoint: outputs\eurosat_mobilenetv2\linear_probe\best.pt
summary: outputs\eurosat_mobilenetv2\linear_probe\summary.json


In [7]:
# partial_unfreeze
run_strategy("partial_unfreeze")

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=0.4915 val_loss=0.2011 val_top1=0.9326 val_f1=0.9297


epoch=2 train_loss=0.2393 val_loss=0.1489 val_top1=0.9479 val_f1=0.9460


epoch=3 train_loss=0.2051 val_loss=0.1360 val_top1=0.9543 val_f1=0.9535


epoch=4 train_loss=0.1799 val_loss=0.1225 val_top1=0.9587 val_f1=0.9578


epoch=5 train_loss=0.1623 val_loss=0.1168 val_top1=0.9626 val_f1=0.9614


epoch=6 train_loss=0.1469 val_loss=0.1156 val_top1=0.9590 val_f1=0.9582


epoch=7 train_loss=0.1410 val_loss=0.1098 val_top1=0.9638 val_f1=0.9625


epoch=8 train_loss=0.1356 val_loss=0.1006 val_top1=0.9663 val_f1=0.9654


strategy=partial_unfreeze test_top1=0.9670 test_f1=0.9662 time=579.1s
best checkpoint: outputs\eurosat_mobilenetv2\partial_unfreeze\best.pt
summary: outputs\eurosat_mobilenetv2\partial_unfreeze\summary.json


In [7]:
# full_finetune
run_strategy("full_finetune")

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=0.3258 val_loss=0.0880 val_top1=0.9710 val_f1=0.9698


epoch=2 train_loss=0.1331 val_loss=0.0755 val_top1=0.9747 val_f1=0.9739


epoch=3 train_loss=0.0985 val_loss=0.0674 val_top1=0.9756 val_f1=0.9746


epoch=4 train_loss=0.0777 val_loss=0.0585 val_top1=0.9811 val_f1=0.9804


epoch=5 train_loss=0.0730 val_loss=0.0979 val_top1=0.9710 val_f1=0.9704


epoch=6 train_loss=0.0663 val_loss=0.0722 val_top1=0.9759 val_f1=0.9753


epoch=7 train_loss=0.0621 val_loss=0.0744 val_top1=0.9781 val_f1=0.9775


epoch=8 train_loss=0.0481 val_loss=0.0604 val_top1=0.9788 val_f1=0.9777


strategy=full_finetune test_top1=0.9821 test_f1=0.9817 time=959.3s
best checkpoint: outputs\eurosat_mobilenetv2\full_finetune\best.pt
summary: outputs\eurosat_mobilenetv2\full_finetune\summary.json


In [5]:
# Run full ablation and always write ablation_results.csv
strategies = ["zero_shot", "from_scratch", "linear_probe", "partial_unfreeze", "full_finetune"]
rows = []
for strategy in strategies:
    rows.append(run_strategy(strategy))

result_csv = OUT_BASE / "ablation_results.csv"
fieldnames = [
    "strategy",
    "best_val_top1",
    "test_top1_acc",
    "test_macro_f1",
    "train_seconds",
    "trainable_params",
    "total_params",
]
with result_csv.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

print("Saved:", result_csv)
for r in rows:
    print(r)

runtime device: cuda:0 (NVIDIA GeForce RTX 4050 Laptop GPU)


epoch=1 train_loss=2.2913 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=2 train_loss=2.2915 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=3 train_loss=2.2924 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


epoch=4 train_loss=2.2916 val_loss=2.3001 val_top1=0.1518 val_f1=0.1101


...

epoch=5 train_loss=0.0739 val_loss=0.0682 val_top1=0.9779 val_f1=0.9770


epoch=6 train_loss=0.0669 val_loss=0.0617 val_top1=0.9811 val_f1=0.9803


epoch=7 train_loss=0.0555 val_loss=0.0592 val_top1=0.9820 val_f1=0.9814


epoch=8 train_loss=0.0551 val_loss=0.0636 val_top1=0.9791 val_f1=0.9783


strategy=full_finetune test_top1=0.9811 test_f1=0.9805 time=951.5s
best checkpoint: outputs\eurosat_mobilenetv2\full_finetune\best.pt
summary: outputs\eurosat_mobilenetv2\full_finetune\summary.json


In [5]:
# Display aggregated results (always rebuild from summary.json)
import json

result_csv = OUT_BASE / "ablation_results.csv"
strategies = ["zero_shot", "from_scratch", "linear_probe", "partial_unfreeze", "full_finetune"]
rows = []

for s in strategies:
    summary_file = OUT_BASE / s / "summary.json"
    if not summary_file.exists():
        continue
    d = json.loads(summary_file.read_text(encoding="utf-8"))
    rows.append({
        "strategy": s,
        "best_val_top1": d.get("best_val_top1"),
        "test_top1_acc": d.get("test_top1_acc"),
        "test_macro_f1": d.get("test_macro_f1"),
        "train_seconds": d.get("train_seconds"),
        "trainable_params": d.get("trainable_params"),
        "total_params": d.get("total_params"),
    })

print("summary files found:", len(rows))
if rows:
    fieldnames = [
        "strategy",
        "best_val_top1",
        "test_top1_acc",
        "test_macro_f1",
        "train_seconds",
        "trainable_params",
        "total_params",
    ]
    with result_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print("Saved:", result_csv)
    for r in rows:
        print(r)
else:
    print("No summary.json found. Run at least one strategy first.")

summary files found: 5
Saved: E:\ANU\COMP6242-group-project\outputs\eurosat_mobilenetv2\ablation_results.csv
{'strategy': 'zero_shot', 'best_val_top1': 0.1518208661417323, 'test_top1_acc': 0.15819116370884453, 'test_macro_f1': 0.11536833349662987, 'train_seconds': 553.3628409000003, 'trainable_params': 0, 'total_params': 2236682}
{'strategy': 'from_scratch', 'best_val_top1': 0.9262357832878594, 'test_top1_acc': 0.9216152668937924, 'test_macro_f1': 0.9186302587231907, 'train_seconds': 959.6917448000004, 'trainable_params': 2236682, 'total_params': 2236682}
{'strategy': 'linear_probe', 'best_val_top1': 0.894548337759934, 'test_top1_acc': 0.8942475943114814, 'test_macro_f1': 0.8900872703356086, 'train_seconds': 532.9844235999999, 'trainable_params': 12810, 'total_params': 2236682}
{'strategy': 'partial_unfreeze', 'best_val_top1': 0.9663440510043948, 'test_top1_acc': 0.9670275590551181, 'test_macro_f1': 0.9662020003287004, 'train_seconds': 577.0395356999998, 'trainable_params': 898890, 'to